# Experiment 7: End-to-End Study of Autoencoders
This notebook implements the Fully Connected AE, Convolutional AE, Denoising AE, and Variational AE as specified in the experiment requirements.

In [1]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, losses, backend as K
import pandas as pd
import shutil
from skimage.metrics import structural_similarity as ssim

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Create images directory
os.makedirs('images', exist_ok=True)

# Utility for saving figures
def save_fig(name):
    plt.savefig(f'images/{name}.pdf', bbox_inches='tight')
    plt.savefig(f'images/{name}.png', bbox_inches='tight')
    plt.close()

def save_tex(df, name):
    df.to_latex(f'images/{name}.tex', index=False)
    print(f"Saved {name}.tex")
    display(df)

# Metrics calculation
def calc_metrics(true, pred):
    true = true.squeeze()
    pred = pred.squeeze()
    mse = np.mean((true - pred)**2)
    mae = np.mean(np.abs(true - pred))
    
    # Calculate SSIM using batch loop for efficiency
    ssim_vals = []
    for i in range(len(true)):
        s = ssim(true[i], pred[i], data_range=1.0)
        ssim_vals.append(s)
    
    return mse, mae, np.mean(ssim_vals)


## 1. Dataset Loading and Preprocessing


In [2]:
# Load MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

# Laboratory Subset (10k train, 2k test)
x_train = x_train[:10000]
y_train = y_train[:10000]
x_test = x_test[:2000]
y_test = y_test[:2000]

# Flattened for FC models
x_train_flat = x_train.reshape(-1, 784)
x_test_flat = x_test.reshape(-1, 784)

# Convolutional shape
x_train_conv = x_train[..., np.newaxis]
x_test_conv = x_test[..., np.newaxis]

print("Train shape:", x_train_conv.shape)
print("Test shape:", x_test_conv.shape)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Train shape: (10000, 28, 28, 1)
Test shape: (2000, 28, 28, 1)


## 2. Fully Connected Autoencoder


In [3]:
# Architecture
fc_input = layers.Input(shape=(784,))
x = layers.Dense(128, activation='relu')(fc_input)
x = layers.Dense(32, activation='relu')(x)
fc_latent = layers.Dense(16, activation='relu')(x)
x = layers.Dense(32, activation='relu')(fc_latent)
x = layers.Dense(128, activation='relu')(x)
fc_output = layers.Dense(784, activation='sigmoid')(x)

fc_ae = models.Model(fc_input, fc_output, name='fc_autoencoder')
fc_ae.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='binary_crossentropy')

# Train
start_time = time.time()
history_fc = fc_ae.fit(x_train_flat, x_train_flat, epochs=20, batch_size=128, 
                       validation_data=(x_test_flat, x_test_flat), verbose=1)
fc_time = time.time() - start_time

fc_preds = fc_ae.predict(x_test_flat).reshape(-1, 28, 28)
fc_mse, fc_mae, fc_ssim = calc_metrics(x_test, fc_preds)

print(f"FC-AE Test Metrics - MSE: {fc_mse:.4f}, MAE: {fc_mae:.4f}, SSIM: {fc_ssim:.4f}")

# Plot 1: Original vs Reconstructed
plt.figure(figsize=(20, 4))
for i in range(10):
    plt.subplot(2, 10, i + 1)
    plt.imshow(x_test[i], cmap='gray')
    plt.axis('off')
    plt.title('Orig')
    
    plt.subplot(2, 10, i + 11)
    plt.imshow(fc_preds[i], cmap='gray')
    plt.axis('off')
    plt.title('Recon')
save_fig('1_fc_ae_reconstruction')

# Plot 2: Training & Validation Loss
plt.figure(figsize=(8, 5))
plt.plot(history_fc.history['loss'], label='Train Loss')
plt.plot(history_fc.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title('FC-AE Loss Curve')
plt.legend()
save_fig('2_fc_ae_loss')


I0000 00:00:1790578415.524240      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790578415.527545      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/20
60/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5111

I0000 00:00:1790578420.337262     130 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


79/79 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 0.3524 - val_loss: 0.2555
Epoch 2/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.2476 - val_loss: 0.2310
Epoch 3/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.2184 - val_loss: 0.2009
Epoch 4/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1940 - val_loss: 0.1827
Epoch 5/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1765 - val_loss: 0.1702
Epoch 6/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1665 - val_loss: 0.1634
Epoch 7/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1598 - val_loss: 0.1582
Epoch 8/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1538 - val_loss: 0.1525
Epoch 9/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1491 - val_loss: 0.1497
Epoch 10/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1460 - val_loss: 0.1470
Epoch 11/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1437 - val_loss: 0.1453
Epoch 12/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1419 - val_loss: 0.1431

## 3. Convolutional Autoencoder


In [4]:
conv_input = layers.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation='relu', padding='same')(conv_input)
x = layers.MaxPooling2D(2, padding='same')(x)
x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
x = layers.MaxPooling2D(2, padding='same')(x)
x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)

# Latent feature map
latent_conv = x

x = layers.UpSampling2D(2)(x)
x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
x = layers.UpSampling2D(2)(x)
conv_output = layers.Conv2D(1, 3, activation='sigmoid', padding='same')(x)

cae = models.Model(conv_input, conv_output, name='conv_autoencoder')
cae.compile(optimizer='adam', loss='binary_crossentropy')

start_time = time.time()
history_cae = cae.fit(x_train_conv, x_train_conv, epochs=15, batch_size=128, 
                      validation_data=(x_test_conv, x_test_conv), verbose=1)
cae_time = time.time() - start_time

cae_preds = cae.predict(x_test_conv).squeeze()
cae_mse, cae_mae, cae_ssim = calc_metrics(x_test, cae_preds)

print(f"CAE Test Metrics - MSE: {cae_mse:.4f}, MAE: {cae_mae:.4f}, SSIM: {cae_ssim:.4f}")

# Plot 3: Comparison
plt.figure(figsize=(15, 6))
for i in range(10):
    plt.subplot(3, 10, i + 1)
    plt.imshow(x_test[i], cmap='gray')
    plt.axis('off')
    if i == 0: plt.title('Original')
        
    plt.subplot(3, 10, i + 11)
    plt.imshow(fc_preds[i], cmap='gray')
    plt.axis('off')
    if i == 0: plt.title('FC-AE')
        
    plt.subplot(3, 10, i + 21)
    plt.imshow(cae_preds[i], cmap='gray')
    plt.axis('off')
    if i == 0: plt.title('CAE')
save_fig('3_fc_cae_comparison')


Epoch 1/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 8s 45ms/step - loss: 0.2330 - val_loss: 0.1028
Epoch 2/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0918 - val_loss: 0.0864
Epoch 3/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0823 - val_loss: 0.0779
Epoch 4/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0785 - val_loss: 0.0753
Epoch 5/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0761 - val_loss: 0.0736
Epoch 6/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0748 - val_loss: 0.0726
Epoch 7/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0736 - val_loss: 0.0733
Epoch 8/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0728 - val_loss: 0.0710
Epoch 9/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0721 - val_loss: 0.0707
Epoch 10/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0716 - val_loss: 0.0698
Epoch 11/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0711 - val_loss: 0.0694
Epoch 12/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0707 - val_l

## 4. Denoising Autoencoder


In [5]:
def add_gaussian_noise(data, sigma):
    noisy = data + np.random.normal(0, sigma, data.shape)
    return np.clip(noisy, 0., 1.)

def add_sp_noise(data, p):
    noisy = np.copy(data)
    mask = np.random.rand(*data.shape)
    noisy[mask < p/2] = 0.
    noisy[(mask >= p/2) & (mask < p)] = 1.
    return noisy

noise_levels = [0.1, 0.2, 0.3]
dae_metrics = []
dae_models = {}

dae_time = 0
for sigma in noise_levels:
    print(f"Training DAE with sigma = {sigma}...")
    xn_train = add_gaussian_noise(x_train_conv, sigma)
    xn_test = add_gaussian_noise(x_test_conv, sigma)
    
    # Clone architecture
    dae = models.clone_model(cae)
    dae.compile(optimizer='adam', loss='binary_crossentropy')
    
    start_time = time.time()
    dae.fit(xn_train, x_train_conv, epochs=10, batch_size=128, verbose=0)
    if sigma == 0.2: dae_time = time.time() - start_time
    
    preds = dae.predict(xn_test).squeeze()
    mse, mae, ssim_val = calc_metrics(x_test, preds)
    dae_metrics.append({'Noise Level': sigma, 'MSE': mse, 'MAE': mae, 'SSIM': ssim_val})
    
    dae_models[sigma] = (dae, preds, xn_test.squeeze())

df_dae = pd.DataFrame(dae_metrics)
save_tex(df_dae, 'dae_noise_metrics')

# Plot 4: DAE Reconstruction for sigma = 0.2
_, dae_02_preds, xn_test_02 = dae_models[0.2]
plt.figure(figsize=(15, 6))
for i in range(10):
    plt.subplot(3, 10, i + 1)
    plt.imshow(x_test[i], cmap='gray')
    plt.axis('off')
    if i == 0: plt.title('Clean')
        
    plt.subplot(3, 10, i + 11)
    plt.imshow(xn_test_02[i], cmap='gray')
    plt.axis('off')
    if i == 0: plt.title('Noisy (σ=0.2)')
        
    plt.subplot(3, 10, i + 21)
    plt.imshow(dae_02_preds[i], cmap='gray')
    plt.axis('off')
    if i == 0: plt.title('Denoised')
save_fig('4_dae_reconstruction')

# Plot 5: Noise Level vs Metrics
fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()
ax1.plot(df_dae['Noise Level'], df_dae['MSE'], 'o-', color='r', label='MSE')
ax1.plot(df_dae['Noise Level'], df_dae['MAE'], 's-', color='g', label='MAE')
ax2.plot(df_dae['Noise Level'], df_dae['SSIM'], '^-', color='b', label='SSIM')

ax1.set_xlabel('Noise Level (Sigma)')
ax1.set_ylabel('MSE / MAE')
ax2.set_ylabel('SSIM')
ax1.legend(loc='center left')
ax2.legend(loc='center right')
plt.title('Reconstruction Quality vs Noise Level')
save_fig('5_noise_vs_metrics')


Training DAE with sigma = 0.1...
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
Training DAE with sigma = 0.2...
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
Training DAE with sigma = 0.3...
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
Saved dae_noise_metrics.tex


,Noise Level,MSE,MAE,SSIM
0,0.1,0.004015,0.019276,0.955849
1,0.2,0.005439,0.023155,0.936513
2,0.3,0.007714,0.028679,0.907470


## 5. Variational Autoencoder (VAE)


In [6]:
latent_dim = 2

class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# Encoder
vae_input = layers.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation='relu', strides=2, padding='same')(vae_input)
x = layers.Conv2D(64, 3, activation='relu', strides=2, padding='same')(x)
x = layers.Flatten()(x)
x = layers.Dense(16, activation='relu')(x)
z_mean = layers.Dense(latent_dim, name='z_mean')(x)
z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)
z = Sampling()([z_mean, z_log_var])
encoder = models.Model(vae_input, [z_mean, z_log_var, z], name='encoder')

# Decoder
latent_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(7 * 7 * 64, activation='relu')(latent_inputs)
x = layers.Reshape((7, 7, 64))(x)
x = layers.Conv2DTranspose(64, 3, activation='relu', strides=2, padding='same')(x)
x = layers.Conv2DTranspose(32, 3, activation='relu', strides=2, padding='same')(x)
vae_output = layers.Conv2DTranspose(1, 3, activation='sigmoid', padding='same')(x)
decoder = models.Model(latent_inputs, vae_output, name='decoder')

# VAE Model
class VAE(models.Model):
    def __init__(self, encoder, decoder, beta=1.0, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta
        self.total_loss_tracker = tf.keras.metrics.Mean(name='total_loss')
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(name='reconstruction_loss')
        self.kl_loss_tracker = tf.keras.metrics.Mean(name='kl_loss')

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(losses.binary_crossentropy(data, reconstruction), axis=(1, 2))
            )
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1)) * self.beta
            total_loss = reconstruction_loss + kl_loss
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            'loss': self.total_loss_tracker.result(),
            'reconstruction_loss': self.reconstruction_loss_tracker.result(),
            'kl_loss': self.kl_loss_tracker.result(),
        }
        
    def test_step(self, data):
        if isinstance(data, tuple): data = data[0]
        z_mean, z_log_var, z = self.encoder(data)
        reconstruction = self.decoder(z)
        reconstruction_loss = tf.reduce_mean(
            tf.reduce_sum(losses.binary_crossentropy(data, reconstruction), axis=(1, 2))
        )
        kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1)) * self.beta
        total_loss = reconstruction_loss + kl_loss
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            'loss': self.total_loss_tracker.result(),
            'reconstruction_loss': self.reconstruction_loss_tracker.result(),
            'kl_loss': self.kl_loss_tracker.result(),
        }

vae = VAE(encoder, decoder)
vae.compile(optimizer='adam')

start_time = time.time()
history_vae = vae.fit(x_train_conv, epochs=20, batch_size=128, validation_data=(x_test_conv, x_test_conv), verbose=1)
vae_time = time.time() - start_time

_, _, z_test = vae.encoder.predict(x_test_conv)
vae_preds = vae.decoder.predict(z_test).squeeze()
vae_mse, vae_mae, vae_ssim = calc_metrics(x_test, vae_preds)

# Additional Training/Validation Loss for VAE (Plot 9b)
plt.figure(figsize=(8, 5))
plt.plot(history_vae.history['reconstruction_loss'], label='Train Rec Loss')
plt.plot(history_vae.history['val_reconstruction_loss'], label='Val Rec Loss')
plt.legend()
plt.title('VAE Reconstruction Loss')
save_fig('9b_vae_loss')


Epoch 1/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 11s 66ms/step - kl_loss: 3.7228 - loss: 299.3295 - reconstruction_loss: 295.6067 - val_kl_loss: 3.0932 - val_loss: 206.9468 - val_reconstruction_loss: 203.8536
Epoch 2/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - kl_loss: 3.0988 - loss: 207.9886 - reconstruction_loss: 204.8898 - val_kl_loss: 3.1359 - val_loss: 196.0476 - val_reconstruction_loss: 192.9117
Epoch 3/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - kl_loss: 3.0592 - loss: 200.4484 - reconstruction_loss: 197.3892 - val_kl_loss: 3.8501 - val_loss: 192.8748 - val_reconstruction_loss: 189.0248
Epoch 4/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - kl_loss: 3.2098 - loss: 198.1190 - reconstruction_loss: 194.9092 - val_kl_loss: 3.7708 - val_loss: 190.8279 - val_reconstruction_loss: 187.0571
Epoch 5/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - kl_loss: 3.3767 - loss: 195.9171 - reconstruction_loss: 192.5404 - val_kl_loss: 3.7163 - val_loss: 189.9839 - val_reconstruction_loss: 186.2677
Epoch 6/20
79/79 ━

## 6. VAE Analysis and Visualization


In [7]:
# Plot 6: Latent Space Visualization
plt.figure(figsize=(8, 6))
scatter = plt.scatter(z_test[:, 0], z_test[:, 1], c=y_test, cmap='tab10', alpha=0.7, s=15)
plt.colorbar(scatter)
plt.title('VAE Latent Space')
plt.xlabel('z1')
plt.ylabel('z2')
save_fig('6_vae_latent_space')

# Plot 7: Randomly Generated Images
random_z = np.random.normal(0, 1, (25, 2))
gen_imgs = vae.decoder.predict(random_z).squeeze()

plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(gen_imgs[i], cmap='gray')
    plt.axis('off')
plt.suptitle('Randomly Generated Images from VAE', y=0.92)
save_fig('7_vae_generated')

# Plot 8: Latent Space Interpolation
z_A = z_test[0] # Try to pick distinct digits if you want, taking random for now
z_B = z_test[3]
alphas = np.linspace(0, 1, 11)
interps = [(1 - a) * z_A + a * z_B for a in alphas]
interp_imgs = vae.decoder.predict(np.array(interps)).squeeze()

plt.figure(figsize=(15, 2))
for i, img in enumerate(interp_imgs):
    plt.subplot(1, 11, i + 1)
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    if i == 0: plt.title('Alpha=0')
    if i == 10: plt.title('Alpha=1')
plt.suptitle('Latent Interpolation', y=1.1)
save_fig('8_latent_interpolation')


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 396ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 381ms/step


## 7. Reconstruction Errors


In [8]:
# Plot 9: Distribution of Reconstruction Errors
errors = np.mean((x_test_flat - fc_preds.reshape(-1, 784))**2, axis=1)

plt.figure(figsize=(8, 5))
plt.hist(errors, bins=50, color='purple', alpha=0.7)
plt.title('Distribution of Reconstruction Errors (FC-AE)')
plt.xlabel('MSE per Image')
plt.ylabel('Frequency')
save_fig('9_error_distribution')

# Plot 10: High-Error Image Analysis
high_err_idx = np.argsort(errors)[-5:]

plt.figure(figsize=(12, 5))
for i, idx in enumerate(high_err_idx):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[idx], cmap='gray')
    plt.axis('off')
    if i == 0: plt.ylabel('Original')
    
    plt.subplot(2, 5, i + 6)
    plt.imshow(fc_preds[idx], cmap='gray')
    plt.axis('off')
    if i == 0: plt.ylabel('Reconstruction')
    plt.title(f"Error: {errors[idx]:.4f}")
save_fig('10_high_error_images')


## 8. Consolidated Results


In [9]:
res = [
    {'Model': 'FC Autoencoder', 'MSE': fc_mse, 'MAE': fc_mae, 'SSIM': fc_ssim, 'Parameters': fc_ae.count_params(), 'Time (s)': round(fc_time, 2)},
    {'Model': 'Conv. Autoencoder', 'MSE': cae_mse, 'MAE': cae_mae, 'SSIM': cae_ssim, 'Parameters': cae.count_params(), 'Time (s)': round(cae_time, 2)},
    {'Model': 'Denoising CAE (σ=0.2)', 'MSE': df_dae[df_dae['Noise Level']==0.2]['MSE'].values[0], 
     'MAE': df_dae[df_dae['Noise Level']==0.2]['MAE'].values[0], 'SSIM': df_dae[df_dae['Noise Level']==0.2]['SSIM'].values[0], 
     'Parameters': dae_models[0.2][0].count_params(), 'Time (s)': round(dae_time, 2)},
    {'Model': 'VAE', 'MSE': vae_mse, 'MAE': vae_mae, 'SSIM': vae_ssim, 'Parameters': vae.encoder.count_params() + vae.decoder.count_params(), 'Time (s)': round(vae_time, 2)}
]

df_res = pd.DataFrame(res)
save_tex(df_res, 'consolidated_results')


Saved consolidated_results.tex


,Model,MSE,MAE,SSIM,Parameters,Time (s)
0,FC Autoencoder,0.024518,0.063569,0.710191,211040,12.12
1,Conv. Autoencoder,0.002990,0.016063,0.970596,74497,17.42
2,Denoising CAE (σ=0.2),0.005439,0.023155,0.936513,74497,9.88
3,VAE,0.050543,0.115674,0.398873,134165,22.71


## 9. Additional Latent-Dimension Study


In [10]:
dims = [2, 8, 16, 32]
dim_metrics = []

for d in dims:
    print(f"Training FC-AE with latent dim = {d}...")
    inputs = layers.Input(shape=(784,))
    x = layers.Dense(128, activation='relu')(inputs)
    x = layers.Dense(32, activation='relu')(x)
    latent = layers.Dense(d, activation='relu')(x)
    x = layers.Dense(32, activation='relu')(latent)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(784, activation='sigmoid')(x)
    
    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy')
    model.fit(x_train_flat, x_train_flat, epochs=10, batch_size=256, verbose=0)
    
    preds = model.predict(x_test_flat).reshape(-1, 28, 28)
    mse, mae, ssim_val = calc_metrics(x_test, preds)
    dim_metrics.append({'Latent Dimension': d, 'MSE': mse, 'SSIM': ssim_val, 'Parameters': model.count_params()})

df_dim = pd.DataFrame(dim_metrics)
save_tex(df_dim, 'latent_dim_study')

# Plot 11: Latent Dimension vs MSE
fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()
ax1.plot(df_dim['Latent Dimension'], df_dim['MSE'], 'o-', color='r', label='MSE')
ax2.plot(df_dim['Latent Dimension'], df_dim['SSIM'], 's-', color='b', label='SSIM')

ax1.set_xlabel('Latent Dimension')
ax1.set_ylabel('MSE')
ax2.set_ylabel('SSIM')
ax1.legend(loc='upper left')
ax2.legend(loc='lower right')
plt.title('Latent Dimension vs Reconstruction Quality')
save_fig('11_latent_dim_vs_mse')


Training FC-AE with latent dim = 2...
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Training FC-AE with latent dim = 8...
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Training FC-AE with latent dim = 16...
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Training FC-AE with latent dim = 32...
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Saved latent_dim_study.tex


,Latent Dimension,MSE,SSIM,Parameters
0,2,0.054750,0.332076,210130
1,8,0.035639,0.581956,210520
2,16,0.037000,0.569067,211040
3,32,0.030779,0.630087,212080


## 10. Zip Images Folder


In [11]:
# Finally, zip the images folder
shutil.make_archive('images_archive', 'zip', 'images')
print("Saved all images and metrics into images_archive.zip!")


Saved all images and metrics into images_archive.zip!
